## ML_1039: Adam Optimizer

**Difficulty**: Medium | **TorchCode**: #29

### Problem
Implement the Adam optimizer from scratch. Maintain first (m) and second (v) moment estimates with exponential decay, apply bias correction, then update parameters.

### Formula
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g, \quad v_t = \beta_2 v_{t-1} + (1-\beta_2)g^2$$
$$\hat{m} = \frac{m_t}{1-\beta_1^t}, \quad \hat{v} = \frac{v_t}{1-\beta_2^t}, \quad \theta \leftarrow \theta - \alpha \frac{\hat{m}}{\sqrt{\hat{v}} + \varepsilon}$$

In [ ]:
import torch

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is None:
                    continue
                self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * p.grad
                self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * p.grad ** 2
                m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                v_hat = self.v[i] / (1 - self.beta2 ** self.t)
                p -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

In [ ]:
import torch

# --- Test 1: Parameters change after step ---
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
(w ** 2).sum().backward()
w_before = w.data.clone()
opt.step()
assert not torch.equal(w.data, w_before)

# --- Test 2: Matches torch.optim.Adam ---
torch.manual_seed(0)
w1 = torch.randn(8, 4, requires_grad=True)
w2 = w1.data.clone().requires_grad_(True)
opt1 = MyAdam([w1], lr=0.001, betas=(0.9, 0.999), eps=1e-8)
opt2 = torch.optim.Adam([w2], lr=0.001, betas=(0.9, 0.999), eps=1e-8)
for _ in range(5):
    (w1 ** 2).sum().backward(); opt1.step(); opt1.zero_grad()
    (w2 ** 2).sum().backward(); opt2.step(); opt2.zero_grad()
assert torch.allclose(w1.data, w2.data, atol=1e-5)

# --- Test 3: zero_grad works ---
w = torch.randn(4, requires_grad=True)
opt = MyAdam([w], lr=0.01)
(w ** 2).sum().backward()
assert w.grad.abs().sum() > 0
opt.zero_grad()
assert w.grad.abs().sum() == 0

print('All tests passed!')